In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, f1_score

df = pd.read_csv("creditcard.csv")

X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

fraud_weight = (len(y_train) - sum(y_train)) / sum(y_train)

In [2]:
def best_threshold(probs, y_true):
    thresholds = np.linspace(0.5, 0.99, 50)
    best_f1 = 0
    best_t = 0.5
    
    for t in thresholds:
        preds = (probs > t).astype(int)
        f1 = f1_score(y_true, preds)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
            
    return best_t, best_f1

In [3]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(class_weight='balanced', max_iter=1000)
lr.fit(X_train, y_train)

probs_lr = lr.predict_proba(X_test)[:,1]

t_lr, f1_lr = best_threshold(probs_lr, y_test)

preds_lr = (probs_lr > t_lr).astype(int)

print("Logistic Regression")
print("Best Threshold:", t_lr)
print("Best F1:", f1_lr)
print(classification_report(y_test, preds_lr))
print("ROC-AUC:", roc_auc_score(y_test, probs_lr))

Logistic Regression
Best Threshold: 0.99
Best F1: 0.680327868852459
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.57      0.85      0.68        98

    accuracy                           1.00     56962
   macro avg       0.78      0.92      0.84     56962
weighted avg       1.00      1.00      1.00     56962

ROC-AUC: 0.9720834996210077


In [4]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42
)

rf.fit(X_train, y_train)

probs_rf = rf.predict_proba(X_test)[:,1]

t_rf, f1_rf = best_threshold(probs_rf, y_test)

preds_rf = (probs_rf > t_rf).astype(int)

print("Random Forest")
print("Best Threshold:", t_rf)
print("Best F1:", f1_rf)
print(classification_report(y_test, preds_rf))
print("ROC-AUC:", roc_auc_score(y_test, probs_rf))

Random Forest
Best Threshold: 0.5
Best F1: 0.8457142857142858
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.96      0.76      0.85        98

    accuracy                           1.00     56962
   macro avg       0.98      0.88      0.92     56962
weighted avg       1.00      1.00      1.00     56962

ROC-AUC: 0.9571936047913819


In [5]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [6]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=fraud_weight,
    random_state=42,
    eval_metric='logloss'
)

xgb.fit(X_train, y_train)

probs_xgb = xgb.predict_proba(X_test)[:,1]
t_xgb, f1_xgb = best_threshold(probs_xgb, y_test)

print("XGBoost Best Threshold:", t_xgb)
print("XGBoost Best F1:", f1_xgb)

preds_xgb = (probs_xgb > t_xgb).astype(int)
print(classification_report(y_test, preds_xgb))
print("ROC-AUC:", roc_auc_score(y_test, probs_xgb))



XGBoost Best Threshold: 0.98
XGBoost Best F1: 0.8539325842696629
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.95      0.78      0.85        98

    accuracy                           1.00     56962
   macro avg       0.97      0.89      0.93     56962
weighted avg       1.00      1.00      1.00     56962

ROC-AUC: 0.9746229456892492


In [7]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

# Scale data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

mlp = MLPClassifier(
    hidden_layer_sizes=(64,32),
    max_iter=300,
    random_state=42
)

mlp.fit(X_train, y_train)

probs_mlp = mlp.predict_proba(X_test)[:,1]

t_mlp, f1_mlp = best_threshold(probs_mlp, y_test)

preds_mlp = (probs_mlp >= t_mlp).astype(int)

print("MLP")
print(classification_report(y_test, preds_mlp))
print("ROC-AUC:", roc_auc_score(y_test, probs_mlp))

MLP
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.91      0.81      0.85        98

    accuracy                           1.00     56962
   macro avg       0.95      0.90      0.93     56962
weighted avg       1.00      1.00      1.00     56962

ROC-AUC: 0.9753192005558554
